sagittal scatter and density

In [38]:
using CSV, DataFrames, Pandas
# convert txt file to csv
cd("/Users/jun/Documents/Project/Orofacial premotor circuits in the adult - RV tracing and Manipulation/Counting and quantification/roi_tables/2020_0521_test")

# read files
wp007 = CSV.read("roi_table_007retro_wp4_20200408.txt");
mas016 = CSV.read("016_mas2.txt");
genio4 = CSV.read("024retro_genio4.txt");

nucleus = "IRN";
file_name = [:wp007 :mas016 :genio4];


In [27]:
# left
wp007 = wp007[wp007.ML_location .< 0, :];
mas016 = mas016[mas016.ML_location .< 0, :];
genio4 = genio4[genio4.ML_location .< 0, :];

In [39]:
# right
wp007 = wp007[wp007.ML_location .> 0, :];
mas016 = mas016[mas016.ML_location .> 0, :];
genio4 = genio4[genio4.ML_location .> 0, :];

In [40]:
# convert the coordinates in the tables to Allen CCF coordinates

for i = 1:length(file_name);
    eval(file_name[i]).AP_location = -(eval(file_name[i]).AP_location*100);
    eval(file_name[i]).ML_location = eval(file_name[i]).ML_location*100;
    eval(file_name[i]).DV_location = eval(file_name[i]).DV_location*100;
end

for i = 1:length(file_name);
    eval(file_name[i]).AP_location = (eval(file_name[i]).AP_location[eval(file_name[i]).AP_location .<780]);
    eval(file_name[i]).AP_location = (eval(file_name[i]).AP_location .+ 540);
    # eval(file_name[i]).DV_location = (eval(file_name[i]).DV_location[eval(file_name[i]).AP_location .<780]);
end

In [29]:
# sagittal
# brain outlines
using NPZ
atlas = npzread("/Users/jun/Documents/MATLAB/Allen/annotation_volume_10um_by_index.npy");

using PyCall, PyPlot
sns = pyimport("seaborn")
pygui(true)
plt.style.use("gadfly") # plot with Gadfly style

NumberOfMerge = 100 # arbitrary value.
AP,DV,ML = size(atlas)
sagittal = Array{Float16, 3}(undef, AP, DV, NumberOfMerge);

using Images
for i =1:NumberOfMerge
    sagittal[:,:,i] = canny(ifelse.(convert(Array{Float16}, atlas[:,:,570 + i])
     .<=1, 0, 1),(0.01,0.0));
     # atlas[:, x + i,:]) x = arbitary value. find a value that gives a good outline
end

sagittal_merge = sum(sagittal, dims = 3);
sagittal_merge = convert(Array{Float64,2}, sagittal_merge[:,:,1]);
sagittal_merge = rotr90(sagittal_merge[:,:,1]); # crop in arbitary AP range
sagittal_merge = sagittal_merge[:,end:-1:1];

In [41]:
# scatter and density single
c = ["#FF62A4" "#D3C93A" "#00BCFD"]
# c = ["#00BCFD" "#D3C93A" "#FF62A4"]
cmap = ["PuRd" "Wistia" "Blues"]
i = 1

while i <= length(file_name);
        # scatter plot
        close()
        fig, ax = plt.subplots(1,1, figsize=(10,10))
        ax.imshow(sagittal_merge[:,:,1], cmap="binary", zorder = 0);
        plt.axis("equal");

        ax.scatter(eval(file_name[i])[eval(file_name[i])[:acronym].== eval(nucleus),:].AP_location,
        eval(file_name[i])[eval(file_name[i])[:acronym].== eval(nucleus),:].DV_location,
        c = c[i],  s =0.5, zorder = 1)

        ax.grid()
        ax.xaxis.set_major_formatter(plt.NullFormatter())
        ax.yaxis.set_major_formatter(plt.NullFormatter())

        savefig(string((file_name[i]), "_",  "_sagittal.png"), dpi = 600, format = "png")

        # density plot
        close()
        fig, ax = plt.subplots(1,1, figsize=(10,10))
        ax.imshow(sagittal_merge[:,:,1], cmap="binary", zorder = 0);
        plt.axis("equal");

        ax = sns.kdeplot(eval(file_name[i])[eval(file_name[i])[:acronym].== eval(nucleus),:].AP_location,
        eval(file_name[i])[eval(file_name[i])[:acronym].== eval(nucleus),:].DV_location,
        cmap = cmap[i], n_levels =4, bw = 18, linewidths = 1.5, zorder = 1)

        ax.grid()
        ax.xaxis.set_major_formatter(plt.NullFormatter())
        ax.yaxis.set_major_formatter(plt.NullFormatter())

        savefig(string((file_name[i]), "_",  "_sagittal_density.png"), dpi = 600, format = "png")
        global i = i + 1

end

┌ Warning: `getindex(df::DataFrame, col_ind::ColumnIndex)` is deprecated, use `df[!, col_ind]` instead.
│   caller = top-level scope at In[41]:14
└ @ Core ./In[41]:14
┌ Warning: `getindex(df::DataFrame, col_ind::ColumnIndex)` is deprecated, use `df[!, col_ind]` instead.
│   caller = top-level scope at In[41]:14
└ @ Core ./In[41]:14
┌ Warning: `getindex(df::DataFrame, col_ind::ColumnIndex)` is deprecated, use `df[!, col_ind]` instead.
│   caller = top-level scope at In[41]:30
└ @ Core ./In[41]:30
┌ Warning: `getindex(df::DataFrame, col_ind::ColumnIndex)` is deprecated, use `df[!, col_ind]` instead.
│   caller = top-level scope at In[41]:30
└ @ Core ./In[41]:30


In [36]:
# scatter merge

i = 1

# plotting parameters
# s (default 2): size of dots

fig, ax = plt.subplots(1,1, figsize=(10,10))
ax.imshow(sagittal_merge[:,:,1], cmap="binary", zorder = 0); 
plt.axis("equal");
        
while i <= length(file_name)
            
    ax.scatter(eval(file_name[i])[eval(file_name[i])[:acronym].== eval(nucleus),:].AP_location,
    eval(file_name[i])[eval(file_name[i])[:acronym].== eval(nucleus),:].DV_location,
    c = c[i],  s = 0.5, zorder = 1)
    global i = i + 1
            
        if i == length(file_name) + 1

            ax.grid()
            ax.xaxis.set_major_formatter(plt.NullFormatter())
            ax.yaxis.set_major_formatter(plt.NullFormatter())

            savefig(string("merge", "_", "_sagittal.png"), dpi = 600, format = "png")
            close()
            
end
end

┌ Warning: `getindex(df::DataFrame, col_ind::ColumnIndex)` is deprecated, use `df[!, col_ind]` instead.
│   caller = top-level scope at In[36]:14
└ @ Core ./In[36]:14
┌ Warning: `getindex(df::DataFrame, col_ind::ColumnIndex)` is deprecated, use `df[!, col_ind]` instead.
│   caller = top-level scope at In[36]:14
└ @ Core ./In[36]:14


In [42]:
# density merge

i = 1

# plotting parameters
# n_levels (default 4): # of contour lines  
# bw (default 12): smoothness of contour lines
# linewidths = (default 0.8):

fig, ax = plt.subplots(1,1, figsize=(10,10))
ax.imshow(sagittal_merge[:,:,1], cmap="binary", zorder = 0);
plt.axis("equal");
                
while i <= length(file_name)
    # density plot
    ax = sns.kdeplot(eval(file_name[i])[eval(file_name[i])[:acronym].== eval(nucleus),:].AP_location,
    eval(file_name[i])[eval(file_name[i])[:acronym].== eval(nucleus),:].DV_location,
    cmap = cmap[i], n_levels = 4, bw = 18, linewidths = 1.5, zorder = 1)
    global i = i + 1
                
        if i == length(file_name) + 1

        ax.grid()
        ax.xaxis.set_major_formatter(plt.NullFormatter())
        ax.yaxis.set_major_formatter(plt.NullFormatter())

        savefig(string("merge", "_",  "_sagittal_density.png"), dpi = 600, format = "png")
        close()
end
end

┌ Warning: `getindex(df::DataFrame, col_ind::ColumnIndex)` is deprecated, use `df[!, col_ind]` instead.
│   caller = top-level scope at In[42]:16
└ @ Core ./In[42]:16
┌ Warning: `getindex(df::DataFrame, col_ind::ColumnIndex)` is deprecated, use `df[!, col_ind]` instead.
│   caller = top-level scope at In[42]:16
└ @ Core ./In[42]:16
